# Ukrainian News Dataset Analysis

This notebook provides comprehensive analysis of the FIdo-AI/ua-news dataset for the hybrid NLP pipeline.

## Contents
1. Dataset Overview
2. Text Length Analysis
3. Category Distribution
4. Language Characteristics
5. Summarization Candidates
6. Model Recommendations

In [ ]:
# Import libraries
import sys
from pathlib import Path

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Ukrainian NLP components
from preprocessing.data_pipeline import UkrainianNewsDataPipeline
from preprocessing.ukrainian_text import UkrainianTextProcessor

# Configure plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Enable Ukrainian text rendering
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Libraries imported successfully!")

## 1. Dataset Loading and Basic Overview

In [ ]:
# Initialize data pipeline
pipeline = UkrainianNewsDataPipeline(data_dir='../data')

# Load dataset
print("Loading Ukrainian news dataset...")
datasets = pipeline.load_dataset()

train_df = datasets['train']
test_df = datasets['test']

print(f"\nDataset loaded successfully!")
print(f"Training set: {len(train_df):,} samples")
print(f"Test set: {len(test_df):,} samples")
print(f"Total: {len(train_df) + len(test_df):,} samples")

In [ ]:
# Basic dataset information
print("Dataset Columns:")
for i, col in enumerate(train_df.columns):
    print(f"{i+1}. {col}")

print("\nFirst few samples:")
display(train_df.head())

print("\nDataset Info:")
print(train_df.info())

In [ ]:
# Analyze dataset using pipeline
analysis = pipeline.analyze_dataset()

print("Dataset Analysis Results:")
print("=" * 50)

for key, value in analysis.items():
    print(f"\n{key.replace('_', ' ').title()}:")
    if isinstance(value, dict):
        for sub_key, sub_value in value.items():
            print(f"  {sub_key}: {sub_value}")
    else:
        print(f"  {value}")

## 2. Text Length Analysis

In [ ]:
# Calculate text lengths
train_df['title_length'] = train_df['title'].astype(str).str.len()
train_df['text_length'] = train_df['text'].astype(str).str.len()
train_df['combined_length'] = train_df['title_length'] + train_df['text_length']

# Text length statistics
length_stats = {
    'Title': train_df['title_length'].describe(),
    'Text': train_df['text_length'].describe(),
    'Combined': train_df['combined_length'].describe()
}

print("Text Length Statistics:")
length_df = pd.DataFrame(length_stats).round(1)
display(length_df)

In [ ]:
# Plot text length distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Title length histogram
axes[0, 0].hist(train_df['title_length'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Title Length Distribution')
axes[0, 0].set_xlabel('Characters')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# Text length histogram
axes[0, 1].hist(train_df['text_length'], bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Text Length Distribution')
axes[0, 1].set_xlabel('Characters')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Combined length histogram
axes[1, 0].hist(train_df['combined_length'], bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Combined Length Distribution')
axes[1, 0].set_xlabel('Characters')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

# Box plot of all lengths
length_data = [train_df['title_length'], train_df['text_length'], train_df['combined_length']]
axes[1, 1].boxplot(length_data, labels=['Title', 'Text', 'Combined'])
axes[1, 1].set_title('Length Comparison (Box Plot)')
axes[1, 1].set_ylabel('Characters')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Category Analysis

In [ ]:
# Category distribution
category_counts = train_df['target'].value_counts()
category_percentages = train_df['target'].value_counts(normalize=True) * 100

print("Category Distribution:")
category_analysis = pd.DataFrame({
    'Count': category_counts,
    'Percentage': category_percentages.round(1)
})
display(category_analysis)

print(f"\nTotal categories: {len(category_counts)}")
print(f"Most common category: {category_counts.index[0]} ({category_counts.iloc[0]:,} samples)")
print(f"Least common category: {category_counts.index[-1]} ({category_counts.iloc[-1]:,} samples)")
print(f"Balance ratio (min/max): {category_counts.min() / category_counts.max():.3f}")

In [ ]:
# Plot category distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
category_counts.plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Category Distribution (Count)', fontsize=14)
axes[0].set_xlabel('Categories')
axes[0].set_ylabel('Number of Articles')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

# Add count labels on bars
for i, v in enumerate(category_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', va='bottom')

# Pie chart
axes[1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Category Distribution (Percentage)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Text length by category
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Box plot of text length by category
train_df.boxplot(column='text_length', by='target', ax=axes[0])
axes[0].set_title('Text Length Distribution by Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Text Length (characters)')
axes[0].tick_params(axis='x', rotation=45)

# Average text length by category
avg_lengths = train_df.groupby('target')['text_length'].mean().sort_values(ascending=False)
avg_lengths.plot(kind='bar', ax=axes[1], color='orange', alpha=0.8)
axes[1].set_title('Average Text Length by Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Average Length (characters)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

# Add value labels
for i, v in enumerate(avg_lengths.values):
    axes[1].text(i, v + 10, f'{v:.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("Average text length by category:")
for category, length in avg_lengths.items():
    print(f"  {category}: {length:.0f} characters")

## 4. Ukrainian Language Characteristics Analysis

In [ ]:
# Initialize Ukrainian text processor
text_processor = UkrainianTextProcessor()

# Sample a subset for language analysis (to speed up processing)
sample_size = min(1000, len(train_df))
sample_df = train_df.sample(n=sample_size, random_state=42)

print(f"Analyzing Ukrainian language characteristics on {sample_size} samples...")

# Extract language features
language_features = []
for _, row in sample_df.iterrows():
    combined_text = f"{row['title']} {row['text']}"
    features = text_processor.extract_features(combined_text)
    language_features.append(features)

features_df = pd.DataFrame(language_features)

print("\nLanguage characteristics analysis complete!")
print("\nFeature statistics:")
display(features_df.describe().round(3))

In [ ]:
# Plot Ukrainian language characteristics
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Cyrillic ratio
axes[0, 0].hist(features_df['cyrillic_ratio'], bins=30, alpha=0.7, color='purple', edgecolor='black')
axes[0, 0].set_title('Cyrillic Character Ratio')
axes[0, 0].set_xlabel('Ratio')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# Word count distribution
axes[0, 1].hist(features_df['word_count'], bins=30, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_title('Word Count Distribution')
axes[0, 1].set_xlabel('Number of Words')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Average word length
axes[0, 2].hist(features_df['avg_word_length'], bins=30, alpha=0.7, color='orange', edgecolor='black')
axes[0, 2].set_title('Average Word Length')
axes[0, 2].set_xlabel('Characters per Word')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].grid(True, alpha=0.3)

# Stopword ratio
axes[1, 0].hist(features_df['stopword_ratio'], bins=30, alpha=0.7, color='red', edgecolor='black')
axes[1, 0].set_title('Ukrainian Stopword Ratio')
axes[1, 0].set_xlabel('Ratio')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

# Unique word ratio
axes[1, 1].hist(features_df['unique_word_ratio'], bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[1, 1].set_title('Unique Word Ratio (Vocabulary Richness)')
axes[1, 1].set_xlabel('Ratio')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

# Punctuation analysis
punct_data = features_df[['exclamation_count', 'question_count']]
punct_data.plot(kind='box', ax=axes[1, 2])
axes[1, 2].set_title('Punctuation Distribution')
axes[1, 2].set_ylabel('Count')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Language quality assessment
print("Ukrainian Language Quality Assessment:")
print("=" * 50)

cyrillic_ratio_mean = features_df['cyrillic_ratio'].mean()
print(f"Average Cyrillic ratio: {cyrillic_ratio_mean:.3f}")

if cyrillic_ratio_mean > 0.8:
    print("✓ High Ukrainian content quality (>80% Cyrillic)")
elif cyrillic_ratio_mean > 0.6:
    print("⚠ Moderate Ukrainian content quality (60-80% Cyrillic)")
else:
    print("✗ Low Ukrainian content quality (<60% Cyrillic)")

avg_word_length = features_df['avg_word_length'].mean()
print(f"\nAverage word length: {avg_word_length:.1f} characters")
print(f"Ukrainian stopword ratio: {features_df['stopword_ratio'].mean():.3f}")
print(f"Vocabulary richness: {features_df['unique_word_ratio'].mean():.3f}")

# Text complexity assessment
complex_texts = features_df[features_df['word_count'] > features_df['word_count'].quantile(0.75)]
print(f"\nComplex texts (top 25% by word count): {len(complex_texts)} samples")
print(f"Average complexity metrics for complex texts:")
print(f"  Words: {complex_texts['word_count'].mean():.0f}")
print(f"  Unique word ratio: {complex_texts['unique_word_ratio'].mean():.3f}")
print(f"  Average word length: {complex_texts['avg_word_length'].mean():.1f}")

## 5. Summarization Candidates Analysis

In [ ]:
# Analyze texts suitable for summarization
summarization_thresholds = [200, 300, 500, 1000, 1500]

print("Summarization Candidates Analysis:")
print("=" * 50)

summarization_stats = []

for threshold in summarization_thresholds:
    candidates = train_df[train_df['text_length'] >= threshold]
    percentage = (len(candidates) / len(train_df)) * 100
    
    summarization_stats.append({
        'threshold': threshold,
        'candidates': len(candidates),
        'percentage': percentage,
        'avg_length': candidates['text_length'].mean() if len(candidates) > 0 else 0
    })
    
    print(f"Threshold {threshold} chars: {len(candidates):,} candidates ({percentage:.1f}%)")

summarization_df = pd.DataFrame(summarization_stats)

# Recommended threshold
recommended_threshold = 500  # Good balance between content and candidate count
recommended_candidates = train_df[train_df['text_length'] >= recommended_threshold]

print(f"\nRecommended threshold: {recommended_threshold} characters")
print(f"Candidates: {len(recommended_candidates):,} ({len(recommended_candidates)/len(train_df)*100:.1f}%)")
print(f"Average length: {recommended_candidates['text_length'].mean():.0f} characters")

In [ ]:
# Plot summarization analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Candidates by threshold
axes[0, 0].plot(summarization_df['threshold'], summarization_df['candidates'], 'o-', linewidth=2, markersize=8)
axes[0, 0].set_title('Summarization Candidates by Threshold')
axes[0, 0].set_xlabel('Minimum Length Threshold (characters)')
axes[0, 0].set_ylabel('Number of Candidates')
axes[0, 0].grid(True, alpha=0.3)

# Percentage by threshold
axes[0, 1].plot(summarization_df['threshold'], summarization_df['percentage'], 's-', 
                color='orange', linewidth=2, markersize=8)
axes[0, 1].set_title('Percentage of Dataset by Threshold')
axes[0, 1].set_xlabel('Minimum Length Threshold (characters)')
axes[0, 1].set_ylabel('Percentage of Dataset (%)')
axes[0, 1].grid(True, alpha=0.3)

# Text length distribution with threshold line
axes[1, 0].hist(train_df['text_length'], bins=50, alpha=0.7, color='lightblue', edgecolor='black')
axes[1, 0].axvline(recommended_threshold, color='red', linestyle='--', linewidth=2, 
                   label=f'Recommended threshold: {recommended_threshold}')
axes[1, 0].set_title('Text Length Distribution with Threshold')
axes[1, 0].set_xlabel('Text Length (characters)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Summarization candidates by category
candidate_by_category = recommended_candidates['target'].value_counts()
candidate_by_category.plot(kind='bar', ax=axes[1, 1], color='green', alpha=0.8)
axes[1, 1].set_title(f'Summarization Candidates by Category\n(Threshold: {recommended_threshold} chars)')
axes[1, 1].set_xlabel('Category')
axes[1, 1].set_ylabel('Number of Candidates')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Sample Text Analysis

In [ ]:
# Show sample texts from each category
print("Sample Texts by Category:")
print("=" * 80)

for category in train_df['target'].unique()[:3]:  # Show first 3 categories
    category_samples = train_df[train_df['target'] == category].sample(n=2, random_state=42)
    
    print(f"\nCategory: {category.upper()}")
    print("-" * 40)
    
    for i, (_, row) in enumerate(category_samples.iterrows(), 1):
        print(f"\nSample {i}:")
        print(f"Title: {row['title'][:100]}..." if len(row['title']) > 100 else f"Title: {row['title']}")
        print(f"Text: {row['text'][:200]}..." if len(row['text']) > 200 else f"Text: {row['text']}")
        print(f"Length: {len(row['text'])} characters")
        
        # Show processed version
        cleaned = text_processor.clean_text(row['text'])
        tokens = text_processor.tokenize(cleaned)
        print(f"Processed tokens (first 10): {tokens[:10]}")

In [ ]:
# Vocabulary analysis
print("Ukrainian Vocabulary Analysis:")
print("=" * 50)

# Sample subset for vocabulary analysis
vocab_sample = train_df.sample(n=min(5000, len(train_df)), random_state=42)

# Extract vocabulary
all_texts = [f"{row['title']} {row['text']}" for _, row in vocab_sample.iterrows()]
vocabulary = text_processor.get_vocabulary(all_texts, min_freq=2)

print(f"Total vocabulary size (min_freq=2): {len(vocabulary):,} words")

# Most common words
common_words = sorted(vocabulary.items(), key=lambda x: x[1], reverse=True)[:20]

print("\nMost common Ukrainian words:")
for i, (word, freq) in enumerate(common_words, 1):
    print(f"{i:2d}. {word:15s} ({freq:,} occurrences)")

# Word length distribution
word_lengths = [len(word) for word in vocabulary.keys()]
avg_word_length = np.mean(word_lengths)

print(f"\nVocabulary statistics:")
print(f"Average word length: {avg_word_length:.1f} characters")
print(f"Shortest word: {min(word_lengths)} characters")
print(f"Longest word: {max(word_lengths)} characters")

## 7. Model Recommendations

In [ ]:
# Generate model recommendations based on analysis
print("MODEL RECOMMENDATIONS FOR UKRAINIAN NEWS PIPELINE")
print("=" * 70)

print("\n🔍 DATASET CHARACTERISTICS:")
print(f"• Total samples: {len(train_df):,} train + {len(test_df):,} test")
print(f"• Categories: {len(train_df['target'].unique())} balanced classes")
print(f"• Average text length: {train_df['text_length'].mean():.0f} characters")
print(f"• Cyrillic content ratio: {cyrillic_ratio_mean:.1%}")
print(f"• Summarization candidates: {len(recommended_candidates):,} ({len(recommended_candidates)/len(train_df)*100:.1f}%)")

print("\n📊 CLASSIFICATION MODEL RECOMMENDATIONS:")
print("1. PRIMARY: BERT Multilingual")
print("   - Best for Ukrainian text understanding")
print("   - Pre-trained on 104 languages including Ukrainian")
print("   - Expected accuracy: 85-90%")
print("   - Model: google-bert/bert-base-multilingual-cased")

print("\n2. BASELINE: Classical ML (SVM + TF-IDF)")
print("   - Fast training and inference")
print("   - Good interpretability")
print("   - Expected accuracy: 75-85%")
print("   - Useful for comparison and fallback")

print("\n📝 SUMMARIZATION MODEL RECOMMENDATIONS:")
print("1. PRIMARY: Ukrainian T5 (ukr-models/uk-summarizer)")
print("   - Specifically trained on Ukrainian text")
print("   - Best abstractive summaries")
print("   - Fallback to mT5 if unavailable")

print("\n2. BACKUP: Extractive Summarization")
print("   - TF-IDF based sentence ranking")
print("   - Fast and reliable")
print("   - Good for longer documents")

print("\n⚙️ PIPELINE CONFIGURATION:")
print(f"• Summarization threshold: {recommended_threshold} characters")
print(f"• Expected summarization rate: {len(recommended_candidates)/len(train_df)*100:.1f}%")
print("• Hybrid approach: Classification + conditional summarization")
print("• MLflow tracking for all experiments")

print("\n🚀 DEPLOYMENT RECOMMENDATIONS:")
print("• Docker with Ukrainian locale support")
print("• AWS Lambda for real-time inference")
print("• S3 for model artifacts and data")
print("• MLflow for model registry and versioning")

print("\n✅ NEXT STEPS:")
print("1. Run training pipeline with recommended models")
print("2. Compare BERT vs classical ML performance")
print("3. Evaluate summarization quality on sample texts")
print("4. Deploy hybrid pipeline to AWS Lambda")
print("5. Monitor performance with MLflow")

## 8. Export Analysis Results

In [ ]:
# Export analysis results
analysis_results = {
    'dataset_info': {
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'categories': list(train_df['target'].unique()),
        'avg_text_length': float(train_df['text_length'].mean()),
        'avg_title_length': float(train_df['title_length'].mean())
    },
    'language_characteristics': {
        'cyrillic_ratio': float(cyrillic_ratio_mean),
        'avg_word_length': float(avg_word_length),
        'vocabulary_size': len(vocabulary),
        'most_common_words': common_words[:10]
    },
    'summarization_analysis': {
        'recommended_threshold': recommended_threshold,
        'candidates_count': len(recommended_candidates),
        'candidates_percentage': float(len(recommended_candidates)/len(train_df)*100),
        'avg_candidate_length': float(recommended_candidates['text_length'].mean())
    },
    'recommendations': {
        'primary_classifier': 'google-bert/bert-base-multilingual-cased',
        'baseline_classifier': 'svm',
        'summarization_model': 'ukr-models/uk-summarizer',
        'summarization_threshold': recommended_threshold
    }
}

# Save to JSON
import json
with open('../data/analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print("Analysis results exported to ../data/analysis_results.json")
print("\nNotebook analysis complete! ✅")